<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Dependencies

# installations of third-party libraries
!pip install requests -q
!pip install pandas -q
!pip install tqdm -q

# first-party imports
import os
import json
import os
import csv
import random
import argparse
import unicodedata
import threading
import time
import sys
import zipfile
from pathlib import Path

# third-party imports
import requests
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import drive


In [ ]:
# @title Paths

# Access GDrive
drive.mount('/content/drive')

# set working diretory
working_dir = "/content/drive/MyDrive/llmserver/task_iclr23/raw/"

# create working directory (if not already existing)
os.makedirs(working_dir, exist_ok=True)
print(f"Ensured working directory exists at: {working_dir}.")

# set dataset directory
dataset_dir = "/content/drive/MyDrive/llmserver/task_iclr23/dataset/"

# create dataset directory (if not already existing)
os.makedirs(dataset_dir, exist_ok=True)
print(f"Ensured dataset directory exists at: {dataset_dir}.")

# OpenReviewAPI
OPEN_REVIEW_API = "https://api.openreview.net/notes?invitation=ICLR.cc/2023/Conference/-/Blind_Submission&details=directReplies"


# Crawl Raw ICLR Data

In [ ]:

def get_iclr_reviews(paper_info_filename=None, review_info_filename=None, decision_info_filename=None):
    offset = 0
    limit = 100  # number of paper per request
    max_limit = float('inf') # maximal amount of requests
    paper_list = [] # create empty paper list
    review_list = [] # create empty review list

    while offset + limit <= max_limit:
        print(f'Scraping review data: {offset}/{max_limit}') # print progress
        print("start request")
        response = requests.get(OPEN_REVIEW_API, params={"offset": offset, "limit": limit}) # save responses
        print("end request")
        if response.status_code != 200: #  check for warning messages
            print("Error:", response.status_code)
            break

        data = response.json()
        if not data or 'notes' not in data or not data['notes']: # check content of response
            break

        for note in data['notes']:
            paper = {} # create empty storage for the following paper information

            paper['uid'] = note['id']
            paper['number'] = note['number']
            paper['title'] = note['content']['title']
            paper['authors'] = note['content']['authors']
            paper['abstract'] = note['content']['abstract']
            paper['pdf'] = note['content']['pdf']
            paper['keywords'] = note['content']['keywords']

            if 'details' in note and 'directReplies' in note['details']: # check for complete information
                paper_list.append(paper) # only then append paper information to the empty list
                for directReplies in note['details']['directReplies']:

                    if directReplies['invitation'].endswith("Official_Review"): # check for reply information

                        review = {}
                        review['uid'] = directReplies['id']
                        review['paper_uid'] = paper['uid']
                        review['paper_title'] = paper['title']
                        review.update(directReplies['content'])

                        review_list.append(review) # only then append the paper information to the empty list

        offset += limit # iterate

    with open(paper_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(paper_list, f) # write paper_info-file
    with open(review_info_filename, 'w', encoding = "utf-8") as f:
        json.dump(review_list, f) # write review_info-file

if __name__ == "__main__":
    get_iclr_reviews(paper_info_filename = working_dir + "ICLR2023paper_raw.json", # create 'paper_info_filename'
                     review_info_filename = working_dir + "ICLR2023review_raw.json") # create 'review_info_filename'

    print("Finished scraping ICLR 2023 reviews!")

# Build ICLR Raw Dataset

In [ ]:

def to_ascii(input_str):  # change the utf-8 characters to ascii
    assert(type(input_str) == str) # assert string input
    normalized = unicodedata.normalize("NFKD", input_str) # unicode normalization
    normalized = normalized.replace('"', "'")  # always use ' instead of " to avoid error in json
    ascii_str = "".join(c for c in normalized if c.isascii()) # drops non-ascii characters
    return ascii_str

random.seed(19260817) # create parser
parser = argparse.ArgumentParser()
parser.add_argument("--max_count", type=int, default=10**9) # states which arguments are expected
parser.add_argument("--input_paper", type=str, default=working_dir+"ICLR2023paper_raw.json")
parser.add_argument("--input_review", type=str, default=working_dir+"ICLR2023review_raw.json")
parser.add_argument("--dataset_name", type=str, required=True)
args = parser.parse_args(["--max_count","2000","--dataset_name","rand2000"]) # select up to 2000 papers and store them in rand2000

# read businesses in full dataset, and then rename "uid" key to "id"
with open(args.input_paper, mode="r", encoding="utf-8") as json_file:
    papers = json.load(json_file)
review_cnt = {} # amount of reviews per paper
for paper in papers:
    paper["id"] = paper["uid"]  # rename "uid" -> "id"
    paper.pop("uid", None) # delete "uid"
    paper["title"] = to_ascii(str(paper["title"]))
    paper["keywords"] = to_ascii(str(paper["keywords"]))
    paper["abstract"] = to_ascii(str(paper["abstract"]))
    review_cnt[paper["id"]] = 0 # initialze count at 0

# read reviews in full dataset, and then rename "text" key to "review" and rename "business_id" key to "belong_id"
with open(args.input_review, mode="r", encoding="utf-8") as json_file:
    reviews = json.load(json_file)

# create review format string
review_format_str = '''Summary Of The Paper:

{}

Strength And Weaknesses:

{}

Clarity, Quality, Novelty And Reproducibility:

{}

Summary Of The Review:

{}
'''

for review in reviews:
    review["belong_id"] = review["paper_uid"]  # rename "paper_uid" -> "belong_id"
    review.pop("paper_uid", None) # delete "paper_uid"
    review["review"] = review_format_str.format(review["summary_of_the_paper"],review["strength_and_weaknesses"],review["clarity,_quality,_novelty_and_reproducibility"],review["summary_of_the_review"]) # apply review format string
    review_cnt[review["belong_id"]] += 1 # increment review count of the paper

paper_selected = []
has = {}
for paper in papers:

    def legal(paper):
        if review_cnt[paper["id"]] < 3: return False # only consider papers with at least three reviews
        return True

    if legal(paper):

        paper_selected.append(paper) # select paper
        has[paper["id"]] = [] # create yet empty id-list

# randomizes and limits the amount of papers selected
random.shuffle(paper_selected)
if len(paper_selected) > args.max_count:
    paper_selected = paper_selected[:args.max_count]

print("# items =", len(paper_selected))
with open(working_dir+"paper_"+args.dataset_name+".json","w",encoding="utf-8") as json_file: # create a json-file with the paper information
    json.dump(paper_selected, json_file)

for i, review in enumerate(reviews):
    if review["belong_id"] in has: # only reviews belonging to a selected paper
        has[review["belong_id"]].append(i) # stores it's current review index i
review_selected = []
for paper in paper_selected:
    for i in has[paper["id"]]:
        reviews[i]["review"] = to_ascii(reviews[i]["review"]) # cleans selected reviews
        review_selected.append(reviews[i]) # adds reviews to selected list

print("# reviews =", len(review_selected))
with open(working_dir+"review_"+args.dataset_name+".json","w",encoding="utf-8") as json_file: # create a json-file with the reviews information
    json.dump(review_selected, json_file)


# Dataset Refinement


In [ ]:
# @title Download Papers

with open(working_dir+"paper_rand2000.json", "r") as json_file: # open paper_rand20000
    papers = json.load(json_file)
with open(working_dir+"review_rand2000.json", "r") as json_file: # open review_rand2000
    reviews = json.load(json_file)

def is_file_larger_than(path,lim): # 10 kb
    try:
        size = os.path.getsize(path) # get the size of respective file
        return size > lim # check if size > lim
    except FileNotFoundError: # if the file does not exist
        return False

if __name__ == "__main__":

    for i, paper in enumerate(tqdm(papers, desc="Downloading Papers")): # show download progress

        path_pdf = working_dir + "files/paper_pdf/" + paper["id"] + ".pdf" # create path for downloaded PDFs

        if not is_file_larger_than(path_pdf, 10*1024): # if size <= lim or file is missing
            time.sleep(0.5) # pause execution for 0.5 sec
            url = "https://openreview.net" + paper["pdf"] # complete URL for respective PDF
            print(i, url)
            response = requests.get(url, proxies={"http": None, "https": None}) # request for given PDF

            if response.status_code == 200: # check request
                with open(path_pdf, "wb") as file:
                    file.write(response.content) # write downloaded binary content (PDF) to the file
            else:
                print("Failed to download the file. Status code:", response.status_code) # print error message
                time.sleep(2) # pause execution for 2 sec
                response = requests.get(url, proxies={"http": None, "https": None}) # request again for given PDF
                if response.status_code == 200: # check status
                    with open(path_pdf, "wb") as file:
                        file.write(response.content) # write downloaded binary content (PDF) to the file
                else:
                    print("Failed again:", response.status_code) # print a final error message
                    raise(Exception)

In [ ]:
# @title ZIP PDFs

path_folder = working_dir + "files/paper_pdf/"
path_zip = path_folder + "paper.zip" # path of the zip-file

file_paths = [] # create empty list for full paths of all to be zipped PDFs
for root, _, files in os.walk(path_folder): # iterate through the directory
    for file in files:
        if file.endswith("zip"): # if the file already ends with .zip...
            print(file)
            raise(Exception) # raise error (safety check to avoid zipping zips)
        file_paths.append(path_folder + file) # the full path is added to the file_paths

with zipfile.ZipFile(path_zip, 'w', zipfile.ZIP_DEFLATED) as zipf: # open in write mode and use DEFLATED compression
    for file in tqdm(file_paths, desc="Zipping files"): # iterate and show progess
        zipf.write(file) # add current file to the zip


In [ ]:
# @title Parse Papers (via Hugging Face)

# --- 1. Mount Drive and Define Function ---

drive.mount('/content/drive')
# Your Google Drive is now mounted at /content/drive

def pdf2grobid(filename, grobid_url="https://kermitt2-grobid.hf.space/api/processFulltextDocument"):
    """Sends a single PDF to the GROBID API and returns the TEI XML text."""
    try:
        with open(filename, 'rb') as file:
            files = {'input': file}
            # Add a timeout for safety, though the server usually handles long processing
            response = requests.post(grobid_url, files=files, timeout=300)

        # Check for non-200 status codes (e.g., rate limit, server error)
        if response.status_code != 200:
            response.raise_for_status()

        return response.text

    except requests.exceptions.RequestException as e:
        print(f"Error processing {filename}: {e}")
        return None # Return None on failure

# --- 2. Define Paths and Run Batch Loop ---

# Use the absolute path from your Google Drive (adjust 'Meine Ablage' as needed)
INPUT_DIR = "/content/drive/MyDrive/llmserver/task_iclr23/raw/files/paper_pdf"
OUTPUT_DIR = "/content/drive/MyDrive/llmserver/task_iclr23/raw/files/paper_raw_colab"

# Ensure the output directory exists
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Loop over all PDF files in the input directory
for filename in os.listdir(INPUT_DIR):
    if filename.endswith(".pdf"):
        # Construct the full input and output file paths
        input_file_path = os.path.join(INPUT_DIR, filename)

        # Replace .pdf with .xml for the output filename
        xml_filename = filename.replace(".pdf", ".xml")
        output_file_path = os.path.join(OUTPUT_DIR, xml_filename)

        print(f"Processing: {filename}...")

        # 1. Call the GROBID API function
        tei_xml = pdf2grobid(input_file_path)

        if tei_xml:
            # 2. Save the resulting XML string
            with open(output_file_path, 'w', encoding='utf-8') as f:
                f.write(tei_xml)
            print(f"  -> Success: Saved to {xml_filename}")

        # Add a short delay to avoid hitting the public API's rate limits
        time.sleep(3)

print("\nBatch processing complete!")

In [ ]:
# @title Parse Papers

parser = argparse.ArgumentParser()
# which context, see folder '../openreview/' for openreview and '../yelp/' for yelp
parser.add_argument("--context", type=str, default="openreview")
# which dataset, use 'rl' for openreview and 'pizza' for yelp.
parser.add_argument("--dataset", type=str, required=True)
args = parser.parse_args(["--dataset", "rand1000"])

with open(working_dir+"paper_"+args.dataset+".json", "r") as json_file:
    papers = json.load(json_file)
with open(working_dir+"review_"+args.dataset+".json", "r") as json_file:
    reviews = json.load(json_file)

if __name__ == "__main__":

    # DO NOT USE PYPDF TO PARSE PAPERS
    # USE SCIENCEBEAM INSTEAD
    # (This is because pypdf returns dummy texts, which will lead the raw paper to be too long)
    if False:

        for i, paper in enumerate(papers):

            if i>5: break

            path_pdf = working_dir + "files/paper_pdf/" + paper["id"] + ".pdf"
            path_raw = working_dir + "files/paper_raw/" + paper["id"] + ".txt"

            if not os.path.exists(path_pdf):
                print("error")
                exit(0)

            def parse_pdf(path_pdf):
                text = ""
                with open(path_pdf, 'rb') as file:
                    pdf_reader = PyPDF2.PdfReader(file)
                    num_pages = len(pdf_reader.pages)
                    for page_num in range(num_pages):
                        page = pdf_reader.pages[page_num]
                        text += page.extract_text()
                return text

            text = parse_pdf(path_pdf)
            with open(path_raw, "w") as file:
                file.write(text)

    else:
        path_zip = working_dir + "files/paper_raw.zip"
        path_folder = working_dir + "files/paper_raw/"
        with zipfile.ZipFile(path_zip, 'r') as zip_ref:
            zip_ref.extractall(path_folder)

In [ ]:
# @title Build Refined Dataset

parser = argparse.ArgumentParser()
# which context, see folder '../openreview/' for openreview and '../yelp/' for yelp
parser.add_argument("--context", type=str, default="openreview")
parser.add_argument("--dataset", type=str, required=True)
args = parser.parse_args(["--dataset", "rand2000"])

with open(working_dir+"paper_"+args.dataset+".json", "r") as json_file:
    papers = json.load(json_file)
with open(working_dir+"review_"+args.dataset+".json", "r") as json_file:
    reviews = json.load(json_file)

mp = {}
for review in reviews:
    if review["belong_id"] not in mp:
        mp[review["belong_id"]] = []
    mp[review["belong_id"]].append(review["review"])

assert(len(mp)==2000)
cnt = 0
for (paper_id, reviews) in mp.items():
    assert(len(reviews) >= 3)
    random.seed(19260817 + cnt)
    cnt += 1
    random.shuffle(reviews)

df = pd.DataFrame(columns=["paper_id", "pdf_url", "abstract", "parsed_text","human_review1","human_review2","human_review3"])
for i, paper in enumerate(tqdm(papers, desc="Processing Papers")):
    pdf_url = "https://openreview.net" + paper["pdf"]
    with open(working_dir + "files/paper_raw/" + paper["id"] + ".txt", "r", encoding="utf-8") as f:
        parsed_text = f.read()
    df.loc[len(df)] = [paper["id"], pdf_url, paper["abstract"], parsed_text, mp[paper["id"]][0],mp[paper["id"]][1],mp[paper["id"]][2]]
# df.to_json(dataset_dir + 'dataset_paper.json', index=False)
df.to_parquet(dataset_dir + 'dataset_paper.parquet', index=False)

In [ ]:
# @title Sanity check

dataset_paper = pd.read_parquet(dataset_dir + "dataset_paper.parquet")

for i in range(len(dataset_paper)):
        row = dataset_paper.loc[i]
        if len(row["parsed_text"]) < 1000:
            print(i, row["paper_id"], len(row["parsed_text"]))